# Unit Test Suite: Environment & Component Verification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ltellado/LAQuer-Audit/blob/main/notebooks/tests/laquer-audit-unit-tests.ipynb)

This notebook executes isolated unit tests for the core modules supporting the project **"Reproducing and Extending LAQuer for Post-Hoc Attribution"**.

### Objectives
1. **Environment Sanity**: Validate platform detection, directory structure, baseline submodule integrity (`eranhirs/LAQuer` pinned at commit `a4affb2`), credential retrieval, and PyTorch hardware acceleration (CUDA/MPS/CPU).
2. **Component Isolation**: Exercise each pipeline stage individually to verify API contracts, string normalization, and verification invariants before running full-scale experimental batches.

---

In [10]:
# If running in Google Colab, clone repo and change into the notebook directory
import os, sys

if "google.colab" in sys.modules and not os.path.exists("/content/LAQuer-Audit"):
    !git clone https://github.com/ltellado/LAQuer-Audit.git /content/LAQuer-Audit
    %cd /content/LAQuer-Audit/notebooks/tests

## Test 1: `LAQuerAudit` Environment Bootstrap (`src/bootstrap.py`)

Validates the static bootstrapping utility:
- **Platform Detection**: Distinguishes between local runtimes (VS Code / CLI) and Google Colab.
- **Repository Root Resolution**: Dynamically discovers root paths and registers `src/` on `sys.path`.
- **Baseline Submodule Integrity**: Confirms the pinned `LAQuer` submodule exists.
- **Credential Access**: Loads `OPENAI_API_KEY` from the environment or Colab Secrets (`userdata`).
- **Device Arbitration**: Resolves the optimal compute device (`cuda`, `mps`, or `cpu`).

**Pass Criteria**: Outputs `✓ Unit test passed for LAQuerAudit!` with no unhandled exceptions.

In [11]:
# Test LAQuerAudit
%run ../../src/bootstrap.py

=== [Unit Test] LAQuerAudit ===
Is Google Colab:  False
Repository Root:  /Users/leo/Workspaces/MSAI/LAQuer-Audit
PyTorch Device:   mps
API Key Detected: True

✓ Unit test passed for LAQuerAudit!



## Test 2: `LAQuerLocalizer` LLM In-Context Localizer (`src/localizer.py`)

Tests the core LLM span-extraction module (Hirsch et al., ACL 2025):
- **Prompt Construction**: Compiles the official instruction prompt (`ins_prompt`) and the 3 LFQA few-shot exemplars with target source documents and claim.
- **Deterministic Inference**: Dispatches to OpenAI chat completion with `temperature=0`.
- **Boundary Stripping**: Normalizes trailing boundary punctuation (`clean_span_boundary`) that commercial chat models append to mid-sentence spans.
- **Extractive Invariant Verification**: Asserts that every extracted span matches at least one source document verbatim as an exact substring.

**Pass Criteria**: Outputs parsed spans, reports `All Verbatim Match: True`, and concludes with `✓ Unit test passed for LAQuerLocalizer!`.

In [12]:
# Test LAQuerLocalizer
%run ../../src/localizer.py

=== [Unit Test] LAQuerLocalizer ===
Executing localization with gpt-4o-mini...
Raw Output:         Reparations compensate victims for abuses suffered.
Parsed Spans:       ['Reparations compensate victims for abuses suffered']
Verbatim Flags:     [True]
All Verbatim Match: True
Prompt Tokens:      577
Completion Tokens:  9

✓ Unit test passed for LAQuerLocalizer!

